# Module 1 - Data Science Concepts
**Smart Traffic Accident Analysis and Severity Prediction** - Abin Shaji Thomas

**What this module covers:** problem statement, objectives, scope, data dictionary, data lifecycle,
memory-safe stratified sampling of the 2.8 GB CSV, data cleaning, **missing GPS handling**,
missing weather imputation, weather grouping, outlier handling and feature engineering.

**Project tasks satisfied here:** *clean accident records* and *handle missing GPS values*
(plus the data preparation that every later module depends on).

Outputs: `data/processed/accidents_sample.parquet`, `data/processed/accidents_clean.parquet`,
`data/sample/sample_5k.csv`, `reports/metrics/dataset_stats.json`, `reports/metrics/gps_missing.json`,
`reports/metrics/data_dictionary.csv`, `reports/metrics/cleaning_log.json` and more.

## 1. Problem statement
Road accidents cause delays, economic loss and injuries. Traffic agencies need to know **where** and
**when** accidents cluster, **which conditions** (weather, light, road features) are linked to more
severe traffic impact, and whether the **severity of a new accident can be predicted** from information
available at the time it is reported.

> In this dataset `Severity` (1-4) measures the **impact on traffic** (length of the delay), not the injury
> severity of the people involved (noted in the dataset documentation and in the upstream project).

## 2. Objectives
1. Clean the US Accidents records and handle missing GPS and weather values.
2. Quantify the influence of weather on accident counts and severity.
3. Identify spatial accident hotspots and temporal patterns.
4. Measure correlations between severity and the other attributes.
5. Train and compare Decision Tree, Random Forest and Logistic Regression severity classifiers.
6. Store and query the data in MongoDB and reproduce key aggregations with Hadoop-style MapReduce.
7. Publish maps, time-series plots and an interactive dashboard.

## 3. Scope
* **In scope:** contiguous US, Feb 2016 - Mar 2023 (whole file profiled), analysis and modelling on a
  ~400,000-row stratified sample (memory limit of a laptop).
* **Out of scope:** injury outcomes, traffic volume (no exposure data, so counts are not rates),
  real-time deployment, multi-node cluster execution.

## 4. Data lifecycle used in this project
| Stage | What happens | Where |
|---|---|---|
| Collect | Kaggle CSV (`sobhanmoosavi/us-accidents`, CC BY-NC-SA 4.0) | `data/raw/` |
| Ingest / profile | chunked full-file pass, data dictionary, % missing | this notebook |
| Sample | stratified by Severity, same fraction per chunk | this notebook |
| Clean / transform | duplicates, strings, GPS, weather, outliers, features | this notebook |
| Store | Parquet (columnar), MongoDB (documents) | `data/processed/`, Module 5 |
| Process | pandas, MapReduce (Hadoop Streaming), optional Spark | Modules 2, 5 |
| Analyse / model | EDA, statistics, ML | Modules 2, 4 |
| Visualise / share | maps, plots, Streamlit dashboard, report | Module 3 |

In [1]:
import sys, time, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import cleaning as C, features as Fe, utils as U

U.ensure_dirs()
pd.set_option("display.max_columns", 60, "display.width", 200)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| interpreter:", Path(sys.executable).parent.parent.name)
print("Raw file:", U.DATA_RAW.relative_to(ROOT), f"{U.DATA_RAW.stat().st_size / 1e9:.2f} GB")

Python 3.14.2 | pandas 3.1.0 | interpreter: .venv
Raw file: data\raw\US_Accidents_March23.csv 3.06 GB


## 5. Pass 1 - full-file profile (chunked, memory-safe)
The file is read in chunks of 500,000 rows. For each chunk we accumulate null counts, Severity / State /
Year counts and a 0.1-degree hotspot grid, so the full 2.8 GB file is never held in memory.

In [2]:
prof = C.profile_full_file(U.DATA_RAW, chunksize=500_000)
TOTAL_ROWS = prof["total_rows"]
print(f"Total rows in the full file: {TOTAL_ROWS:,}  (pass took {prof['seconds']} s)")
print("Full-data Severity counts:"); print(prof["severity_counts"].to_string())

Total rows in the full file: 7,728,394  (pass took 30.2 s)
Full-data Severity counts:
Severity
1      67366
2    6156981
3    1299337
4     204710


### Data dictionary (dtype from the first chunk, % missing from the full file)

In [3]:
MEANING = {
    "ID": "Unique record identifier", "Source": "Data provider (API) that reported the accident",
    "Severity": "Impact on traffic, 1 (least) to 4 (most)", "Start_Time": "Local start time of the accident",
    "End_Time": "Local time when the impact on traffic ended", "Start_Lat": "GPS latitude of start point",
    "Start_Lng": "GPS longitude of start point", "End_Lat": "GPS latitude of end point",
    "End_Lng": "GPS longitude of end point", "Distance(mi)": "Length of road extent affected (miles)",
    "Description": "Free-text description of the accident", "Street": "Street name", "City": "City",
    "County": "County", "State": "State (2-letter code)", "Zipcode": "ZIP code", "Country": "Country",
    "Timezone": "Time zone", "Airport_Code": "Nearest airport weather station",
    "Weather_Timestamp": "Time of the weather observation", "Temperature(F)": "Temperature (F)",
    "Wind_Chill(F)": "Wind chill (F)", "Humidity(%)": "Relative humidity (%)", "Pressure(in)": "Air pressure (inches)",
    "Visibility(mi)": "Visibility (miles)", "Wind_Direction": "Wind direction", "Wind_Speed(mph)": "Wind speed (mph)",
    "Precipitation(in)": "Precipitation amount (inches)", "Weather_Condition": "Weather condition text",
    "Amenity": "Amenity nearby", "Bump": "Speed bump nearby", "Crossing": "Crossing nearby",
    "Give_Way": "Give-way sign nearby", "Junction": "Junction nearby", "No_Exit": "No-exit nearby",
    "Railway": "Railway nearby", "Roundabout": "Roundabout nearby", "Station": "Station nearby",
    "Stop": "Stop sign nearby", "Traffic_Calming": "Traffic calming nearby", "Traffic_Signal": "Traffic signal nearby",
    "Turning_Loop": "Turning loop nearby", "Sunrise_Sunset": "Day/Night by sunrise/sunset",
    "Civil_Twilight": "Day/Night by civil twilight", "Nautical_Twilight": "Day/Night by nautical twilight",
    "Astronomical_Twilight": "Day/Night by astronomical twilight",
}
dd = pd.DataFrame({"column": prof["dtypes"].index, "meaning": [MEANING.get(c, "") for c in prof["dtypes"].index],
                   "dtype": prof["dtypes"].values, "missing": prof["nulls"].reindex(prof["dtypes"].index).values})
dd["pct_missing"] = (100 * dd["missing"] / TOTAL_ROWS).round(3)
dd.to_csv(U.METRICS / "data_dictionary.csv", index=False)
dd

,column,meaning,dtype,missing,pct_missing
0,ID,Unique record identifier,str,0,0.000
1,Source,Data provider (API) that reported the accident,str,0,0.000
2,Severity,"Impact on traffic, 1 (least) to 4 (most)",int64,0,0.000
3,Start_Time,Local start time of the accident,str,0,0.000
4,End_Time,Local time when the impact on traffic ended,str,0,0.000
5,Start_Lat,GPS latitude of start point,float64,0,0.000
6,Start_Lng,GPS longitude of start point,float64,0,0.000
7,End_Lat,GPS latitude of end point,float64,3402762,44.029
8,End_Lng,GPS longitude of end point,float64,3402762,44.029
9,Distance(mi),Length of road extent affected (miles),float64,0,0.000


In [4]:
# Full-data counts kept for later modules (hotspot grid is untracked, the small tables are metrics)
prof["grid"].to_parquet(U.DATA_PROCESSED / "full_hotspot_grid.parquet", index=False)
prof["state_counts"].rename_axis("State").rename("accidents").reset_index().to_csv(U.METRICS / "full_state_counts.csv", index=False)
prof["year_counts"].rename_axis("Year").rename("accidents").reset_index().to_csv(U.METRICS / "full_year_counts.csv", index=False)
prof["severity_counts"].rename_axis("Severity").rename("accidents").reset_index().to_csv(U.METRICS / "full_severity_counts.csv", index=False)
print("Columns with > 10% missing:"); print(dd.loc[dd.pct_missing > 10, ["column", "pct_missing"]].to_string(index=False))

Columns with > 10% missing:
           column  pct_missing
          End_Lat       44.029
          End_Lng       44.029
    Wind_Chill(F)       25.866
Precipitation(in)       28.513


## 6. Pass 2 - stratified chunked sample (~400,000 rows)
`Description` is excluded (free text, the largest column, and it describes the outcome). In every chunk the same
fraction `400,000 / total_rows` is drawn **within each Severity level**, with one seeded generator
(`random_state=42`), so the sample keeps the full-data class mix.

In [5]:
TARGET = 400_000
FRAC = TARGET / TOTAL_ROWS
usecols = [c for c in prof["dtypes"].index if c != "Description"]
t0 = time.time()
sample = C.stratified_chunk_sample(U.DATA_RAW, frac=FRAC, usecols=usecols, chunksize=500_000, seed=U.RANDOM_STATE)
print(f"Sample rows: {len(sample):,}  fraction={FRAC:.6f}  ({time.time() - t0:.0f} s)")
sample.to_parquet(U.SAMPLE_PARQUET, index=False)

mix = pd.DataFrame({"full_pct": (100 * prof["severity_counts"] / TOTAL_ROWS).round(3),
                    "sample_pct": (100 * sample["Severity"].value_counts(normalize=True).sort_index()).round(3)})
U.save_json({"total_rows": TOTAL_ROWS, "total_columns": len(prof["dtypes"]), "sample_rows": len(sample),
             "sample_fraction": FRAC, "chunksize": 500_000, "random_state": U.RANDOM_STATE,
             "full_severity_counts": prof["severity_counts"].to_dict(),
             "sample_severity_counts": sample["Severity"].value_counts().sort_index().to_dict(),
             "date_min": str(pd.to_datetime(sample["Start_Time"], format="mixed").min()),
             "date_max": str(pd.to_datetime(sample["Start_Time"], format="mixed").max())},
            U.METRICS / "dataset_stats.json")
mix

Sample rows: 399,999  fraction=0.051757  (25 s)


,full_pct,sample_pct
Severity,,
1,0.872,0.872
2,79.667,79.667
3,16.813,16.813
4,2.649,2.649


## 7. Cleaning
### 7.1 Parse times, remove duplicates, standardise strings
Times are stored in mixed formats (some have fractional seconds), so `format="mixed"` is used.

In [6]:
df = sample.copy()
log = {}
df = C.parse_times(df)
log["unparseable_start_time"] = int(df["Start_Time"].isna().sum())
df = df.dropna(subset=["Start_Time"])
df, log["duplicates"] = C.drop_duplicates(df)
df = C.standardize_strings(df)
print(log)

{'unparseable_start_time': 0, 'duplicates': {'rows_before': 399999, 'duplicate_ids': 0, 'duplicate_content': 418, 'rows_after': 399581}}


### 7.2 Drop leaky / useless columns (justification computed from the data)

In [7]:
just = {
    "Country": f"constant: {df['Country'].nunique()} unique value(s)",
    "Turning_Loop": f"constant: {df['Turning_Loop'].nunique()} unique value(s)",
    "Wind_Chill(F)": f"{100 * df['Wind_Chill(F)'].isna().mean():.1f}% missing and Pearson r="
                     f"{df[['Wind_Chill(F)', 'Temperature(F)']].corr().iloc[0, 1]:.3f} with Temperature (redundant)",
    "Civil_Twilight": f"agrees with Sunrise_Sunset in {100 * (df['Civil_Twilight'] == df['Sunrise_Sunset']).mean():.1f}% of rows (redundant)",
    "Nautical_Twilight": f"agrees with Sunrise_Sunset in {100 * (df['Nautical_Twilight'] == df['Sunrise_Sunset']).mean():.1f}% of rows (redundant)",
    "Astronomical_Twilight": f"agrees with Sunrise_Sunset in {100 * (df['Astronomical_Twilight'] == df['Sunrise_Sunset']).mean():.1f}% of rows (redundant)",
    "Weather_Timestamp": "metadata of the weather observation, not of the accident",
    "Airport_Code": "weather-station id, metadata",
    "Timezone": f"{df['Timezone'].nunique()} values, fully determined by State/longitude",
    "Zipcode": f"{df['Zipcode'].nunique():,} unique values, high-cardinality identifier",
    "Street": f"{df['Street'].nunique():,} unique free-text values",
    "County": f"{df['County'].nunique():,} unique values; State and City are kept instead",
    "Description": "excluded at load: free text written after the event (would leak severity)",
}
drop_cols = [c for c in just if c in df.columns]
df = df.drop(columns=drop_cols)
log["dropped_columns"] = just
pd.DataFrame({"dropped column": list(just), "justification": list(just.values())})

,dropped column,justification
0,Country,constant: 1 unique value(s)
1,Turning_Loop,constant: 1 unique value(s)
2,Wind_Chill(F),25.9% missing and Pearson r=0.994 with Tempera...
3,Civil_Twilight,agrees with Sunrise_Sunset in 95.1% of rows (r...
4,Nautical_Twilight,agrees with Sunrise_Sunset in 90.2% of rows (r...
5,Astronomical_Twilight,agrees with Sunrise_Sunset in 86.2% of rows (r...
6,Weather_Timestamp,"metadata of the weather observation, not of th..."
7,Airport_Code,"weather-station id, metadata"
8,Timezone,"4 values, fully determined by State/longitude"
9,Zipcode,"107,716 unique values, high-cardinality identi..."


Kept but **not used as ML features**: `ID` (identifier, used as MongoDB key), `Source` (data-provider artefact),
`End_Time` / `Duration_min` and `Distance(mi)` (only known after the accident -> leakage; Distance is tested in
Module 4 with and without), `Year` (data-collection artefact, checked in Module 2).

### 7.3 Physically impossible weather values -> missing
Plausible ranges reused from the upstream project (Temperature -60..130 F, Humidity 0..100 %, Pressure 20..35 in,
Visibility 0..100 mi, Wind 0..200 mph, Precipitation 0..20 in). Out-of-range values become NaN and are imputed below.

In [8]:
log["impossible_values_set_to_nan"] = C.plausible_range_to_nan(df)
log["impossible_values_set_to_nan"]

{'Temperature(F)': 7,
 'Humidity(%)': 0,
 'Pressure(in)': 30,
 'Visibility(mi)': 2,
 'Wind_Speed(mph)': 1,
 'Precipitation(in)': 1}

## 8. Missing GPS handling
**Audit:** missing or outside the contiguous-US bounding box (lat 24.39..49.39, lng -124.85..-66.88).

**Rules:**
1. Out-of-bounds coordinates are treated as missing.
2. Missing **Start** GPS -> median Start_Lat/Start_Lng of accidents in the same City + State; if no such city exists,
   the row is dropped (a location-less accident cannot be mapped or clustered).
3. Missing **End** GPS -> copied from the Start point. *Assumption:* the accident is treated as a point event; the
   flag `End_GPS_Imputed` records which rows were imputed so analyses can exclude them.

In [9]:
df, gps = C.handle_missing_gps(df)
gps_out = {"rows_before": log["duplicates"]["rows_after"], **gps}
U.save_json(gps_out, U.METRICS / "gps_missing.json")
pd.DataFrame({
    "Start - missing": [gps["before"]["Start"]["missing"], gps["after"]["Start"]["missing"]],
    "Start - out of US bounds": [gps["before"]["Start"]["out_of_us_bounds"], gps["after"]["Start"]["out_of_us_bounds"]],
    "End - missing": [gps["before"]["End"]["missing"], gps["after"]["End"]["missing"]],
    "End - out of US bounds": [gps["before"]["End"]["out_of_us_bounds"], gps["after"]["End"]["out_of_us_bounds"]],
}, index=["before", "after"])

,Start - missing,Start - out of US bounds,End - missing,End - out of US bounds
before,0,0,176088,0
after,0,0,0,0


In [10]:
print(f"Start GPS recovered from City+State median: {gps['start_recovered_city_state_median']:,}")
print(f"Rows dropped (Start GPS unrecoverable):     {gps['start_dropped_unrecoverable']:,}")
print(f"End GPS imputed from Start:                 {gps['end_imputed_from_start']:,} "
      f"({100 * gps['end_imputed_from_start'] / gps['rows_after']:.1f}% of rows)")
# How do Distance values look where End GPS was missing? (checks the point-event assumption)
print(df.groupby("End_GPS_Imputed")["Distance(mi)"].describe().round(3))

Start GPS recovered from City+State median: 0
Rows dropped (Start GPS unrecoverable):     0
End GPS imputed from Start:                 176,088 (44.1% of rows)
                    count   mean    std  min   25%    50%    75%      max
End_GPS_Imputed                                                          
False            223493.0  0.843  1.904  0.0  0.07  0.284  0.899  118.468
True             176088.0  0.217  1.887  0.0  0.00  0.000  0.000  441.750


## 9. Missing weather values
### 9.1 Testing the upstream rule "missing Precipitation -> 0"
If precipitation is usually missing because nothing fell, observed precipitation in dry weather should be ~0.

In [11]:
before_missing = df[U.WEATHER_NUM + ["Weather_Condition", "Wind_Direction"]].isna().sum()
grp_tmp = Fe.weather_group(df["Weather_Condition"])
precip_test = (df.assign(WeatherGroup=grp_tmp.where(df["Weather_Condition"].notna(), "(missing)"))
               .groupby("WeatherGroup")["Precipitation(in)"]
               .agg(rows="size", pct_missing=lambda s: round(100 * s.isna().mean(), 1),
                    observed_median="median", observed_mean="mean", pct_zero_when_observed=lambda s: round(100 * (s.dropna() == 0).mean(), 1)))
precip_test["observed_mean"] = precip_test["observed_mean"].round(4)
precip_test.to_csv(U.METRICS / "precipitation_rule_test.csv")
precip_test

,rows,pct_missing,observed_median,observed_mean,pct_zero_when_observed
WeatherGroup,,,,,
(missing),8998,84.2,0.00,0.0200,91.5
Clear,175945,26.1,0.00,0.0007,99.7
Cloudy,163861,34.1,0.00,0.0030,95.5
Fog/Haze,10321,25.1,0.00,0.0014,96.8
Other,164,1.2,0.00,0.0060,84.6
Rain,26450,5.2,0.02,0.0530,34.4
Snow,9252,4.8,0.00,0.0127,62.7
Storm,4590,5.7,0.00,0.1091,50.9


**Decision (tested above):** the upstream rule holds only for dry weather groups. Missing precipitation is set
to 0 for Clear / Cloudy / Fog/Haze / Other, and to the State+Month median for Rain / Snow / Storm (where it is
usually non-zero). Other numeric weather -> State+Month median; categorical weather -> State+Month mode.

In [12]:
cat_rep = C.impute_weather_categorical(df)
df["WeatherGroup"] = Fe.weather_group(df["Weather_Condition"])
num_rep = C.impute_weather_numeric(df, dry_mask=~df["WeatherGroup"].isin(Fe.WET_GROUPS))
weather_rep = pd.concat([num_rep, cat_rep], ignore_index=True)
weather_rep["pct_missing_before"] = (100 * weather_rep["missing_before"] / len(df)).round(2)
weather_rep.to_csv(U.METRICS / "weather_imputation.csv", index=False)
weather_rep

,column,missing_before,missing_after,method,pct_missing_before
0,Temperature(F),8591,0,State+Month median,2.15
1,Humidity(%),9140,0,State+Month median,2.29
2,Pressure(in),7369,0,State+Month median,1.84
3,Visibility(mi),9204,0,State+Month median,2.30
4,Wind_Speed(mph),29657,0,State+Month median,7.42
5,Precipitation(in),114071,0,"0 if dry weather group, else State+Month median",28.55
6,Weather_Condition,8998,0,State+Month mode,2.25
7,Wind_Direction,9109,0,State+Month mode,2.28


### 9.2 Weather grouping
More than 100 raw `Weather_Condition` labels are mapped to 7 groups with ordered regex rules
(Storm > Snow > Rain > Fog/Haze > Cloudy > Clear > Other).

In [13]:
mapping = (df.groupby(["WeatherGroup", "Weather_Condition"]).size().rename("rows").reset_index()
           .sort_values(["WeatherGroup", "rows"], ascending=[True, False]))
print("Raw labels:", df["Weather_Condition"].nunique())
print(df["WeatherGroup"].value_counts().to_string())
mapping.groupby("WeatherGroup").head(3)

Raw labels: 100
WeatherGroup
Clear       184327
Cloudy      164458
Rain         26457
Fog/Haze     10321
Snow          9264
Storm         4590
Other          164


,WeatherGroup,Weather_Condition,rows
1,Clear,Fair,140672
0,Clear,Clear,41778
2,Clear,Fair / Windy,1877
5,Cloudy,Mostly Cloudy,53219
3,Cloudy,Cloudy,42210
8,Cloudy,Partly Cloudy,36231
14,Fog/Haze,Fog,5067
16,Fog/Haze,Haze,3899
27,Fog/Haze,Smoke,650
31,Other,N/A Precipitation,163


## 10. Outlier handling
IQR fences are a poor fit for very skewed variables (e.g. Visibility is 10 mi for most rows, so its IQR is ~0 and
the fences would clip almost every non-10 value). The IQR outlier count is shown for every column as a diagnostic,
and **winsorizing at the 0.5th / 99.5th percentiles** is applied instead: extreme but possible values are capped,
not deleted.

In [14]:
out_cols = ["Distance(mi)", "Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)"]
iqr_diag = {}
for c in out_cols:
    q1, q3 = df[c].quantile([0.25, 0.75]); iqr = q3 - q1
    iqr_diag[c] = int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum())
win = C.winsorize(df, out_cols, method="quantile", q=(0.005, 0.995))
win.insert(4, "iqr_outliers", win["column"].map(iqr_diag))
win.to_csv(U.METRICS / "outliers_winsorize.csv", index=False)
win

,column,method,lower,upper,iqr_outliers,n_clipped,mean_before,mean_after,std_before,std_after,max_before,max_after
0,Distance(mi),quantile,0.00,9.669,49741,1998,0.567,0.527,1.922,1.256,441.75,9.669
1,Temperature(F),quantile,6.00,99.000,2596,3692,61.635,61.658,18.967,18.794,129.20,99.000
2,Humidity(%),quantile,9.00,100.000,0,1667,64.872,64.883,22.637,22.610,100.00,100.000
3,Pressure(in),quantile,24.16,30.510,23758,3810,29.538,29.543,1.005,0.970,31.15,30.510
4,Visibility(mi),quantile,0.25,10.000,76360,2845,9.112,9.059,2.640,2.310,100.00,10.000
5,Wind_Speed(mph),quantile,0.00,25.000,15042,1757,7.612,7.591,5.101,5.002,127.00,25.000
6,Precipitation(in),quantile,0.00,0.230,27530,1974,0.006,0.004,0.083,0.024,10.04,0.230


## 11. Feature engineering
`Hour`, `DayOfWeek` (0=Mon), `Month`, `Year`, `IsWeekend`, `IsRushHour` (07-09 and 16-18 on weekdays), `DayNight`,
`RoadFeatureCount` (number of the 13 road-feature flags that are true) and `Duration_min`.
`Duration_min` is **post-event** (needs End_Time): used for EDA only, never as an ML feature. Durations outside
(0, 1440] minutes are set to missing (rule from upstream clean.py).

In [15]:
df = Fe.add_time_features(df)
df = Fe.add_road_feature_count(df)
bad_dur = ~df["Duration_min"].between(0, 1440, inclusive="right")
log["duration_outside_0_1440_set_nan"] = int(bad_dur.sum())
df.loc[bad_dur, "Duration_min"] = np.nan
df["City"] = df["City"].fillna("Unknown")

df = df.drop(columns=["Sunrise_Sunset"])
df[["Hour", "DayOfWeek", "Month", "Year", "IsWeekend", "IsRushHour", "RoadFeatureCount", "Duration_min"]].describe().round(2)

,Hour,DayOfWeek,Month,Year,IsWeekend,IsRushHour,RoadFeatureCount,Duration_min
count,399581.00,399581.00,399581.00,399581.00,399581.00,399581.00,399581.00,397802.00
mean,12.32,2.59,6.71,2019.91,0.16,0.36,0.42,108.15
std,5.46,1.81,3.64,1.92,0.37,0.48,0.73,123.70
min,0.00,0.00,1.00,2016.00,0.00,0.00,0.00,3.00
25%,8.00,1.00,3.00,2018.00,0.00,0.00,0.00,31.13
50%,13.00,3.00,7.00,2020.00,0.00,0.00,0.00,74.73
75%,17.00,4.00,10.00,2022.00,0.00,1.00,1.00,124.55
max,23.00,6.00,12.00,2023.00,1.00,1.00,6.00,1440.00


## 12. Save cleaned data

In [16]:
remaining = df.isna().sum()
print("Remaining missing values:"); print(remaining[remaining > 0].to_string())
df = df.reset_index(drop=True)
df.to_parquet(U.CLEAN_PARQUET, index=False)
df.sample(5000, random_state=U.RANDOM_STATE).to_csv(U.DATA_SAMPLE / "sample_5k.csv", index=False)
log.update({"rows_sample": len(sample), "rows_clean": len(df), "columns_clean": df.shape[1],
            "clean_columns": list(df.columns)})
U.save_json(log, U.METRICS / "cleaning_log.json")
print(f"Clean rows: {len(df):,}  columns: {df.shape[1]}")
print(f"Parquet size: {U.CLEAN_PARQUET.stat().st_size / 1e6:.1f} MB  vs raw CSV {U.DATA_RAW.stat().st_size / 1e6:,.0f} MB (full file)")

Remaining missing values:
Duration_min    1779


Clean rows: 399,581  columns: 45
Parquet size: 28.8 MB  vs raw CSV 3,058 MB (full file)


## Module 1 summary
The full file was profiled in chunks, a Severity-stratified ~400k sample was drawn, and the sample was cleaned
(duplicates, strings, impossible values, GPS, weather, outliers) and enriched with time/road features.
All counts above are saved in `reports/metrics/` (`dataset_stats.json`, `gps_missing.json`, `cleaning_log.json`,
`weather_imputation.csv`, `outliers_winsorize.csv`, `precipitation_rule_test.csv`, `data_dictionary.csv`).